<a href="https://colab.research.google.com/github/legna7816/ml-projects/blob/main/rag/rag_step01_Retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# !pip install sentence-transformers # 최초 1회 설치

import numpy as np
from sentence_transformers import SentenceTransformer

In [2]:
# 1. 문장 임베딩 모델 불러오기
# 이전에 쓴 BERT는 '단어' 벡터를 다루는데 최적화했다면,
# SentenceTransformer는 '문장 전체'를 하나의 벡터로 표현하는 데 특화된 모델
# (전 파트에 겪은 "단순 평균 pooling의 함정"을 이 모델은 학습 단게에서 이미 보완함)
# model = SentenceTransformer('all-MiniLM-L6-v2') # 가볍고 빠른 문장 임베딩 모델

# model = SentenceTransformer('jhgan/ko-sroberta-multitask') # 한국어 지원
# 또는
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')  # 다국어 지원

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [3]:
# 2. 검색 대상 문서 준비 (가짜 베이스)
# 실제로는 회사 메뉴얼, 논문, 위키 문서 등이 들어갈 자리
documents = [
    "타이타닉은 1912년 4월 15일 빙산과 충돌해 침몰한 영국의 여객선이다.",
    "파이썬은 1991년 귀도 반 로섬이 개발한 프로그래밍 언어이다.",
    "BERT는 구글이 2018년에 발표한 자연어처리 모델이다.",
    "김치는 발효 채소를 이용한 한국의 전통 음식이다.",
    "RAG는 검색과 생성을 결합한 자연어처리 기법이다.",
    "딥러닝은 인공신경망을 여러 층으로 쌓아 학습하는 머신러닝의 한 분야이다.",
]

In [4]:
# 3. 문서들을 벡터로 변환 (임베딩)
doc_embeddings = model.encode(documents)
print('문서 벡터 크기:', doc_embeddings.shape) # (문서 개수, 384차원)

문서 벡터 크기: (6, 384)


In [5]:
# 4. 코사인 유사도 함수
def cosine_sim(a, b):
  return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [6]:
# 5. 검색 함수
def search(query, top_k=2):
  query_vec = model.encode(query) # 질문도 똑같은 방식으로 벡터화
  scores = [cosine_sim(query_vec, doc_vec) for doc_vec in doc_embeddings]

  # 유사도 높은 순으로 정렬해서 top_k개 인덱스 추출
  top_indices = np.argsort(scores)[::-1][:top_k]

  return [(documents[i], scores[i]) for i in top_indices]

In [7]:
# 6. 실제 검색 테스트
query1 = "자연어 처리 모델에는 뭐가 있어?"
results1 = search(query1)
print(f'질문: {query1}')
for doc, score in results1:
  print(f'    유사도 {score: .3f}: {doc}')

print()

query2 = "한국 음식 알려줘"
results2 = search(query2)
print(f'질문: {query2}')
for doc, score in results2:
  print(f'    유사도 {score:.3f}: {doc}')

질문: 자연어 처리 모델에는 뭐가 있어?
    유사도  0.444: BERT는 구글이 2018년에 발표한 자연어처리 모델이다.
    유사도  0.427: RAG는 검색과 생성을 결합한 자연어처리 기법이다.

질문: 한국 음식 알려줘
    유사도 0.728: 김치는 발효 채소를 이용한 한국의 전통 음식이다.
    유사도 0.175: BERT는 구글이 2018년에 발표한 자연어처리 모델이다.


In [8]:
# 7. TODO
# 7-1. "배가 침몰한 사건" 질문에 타이타닉 문서가 top1로 뜨는지 확인
query3 = "배가 침몰한 사건"
results3 = search(query3)
print(f'질문: {query3}')
for doc, score in results3:
  print(f'    유사도 {score:.3f}: {doc}')

질문: 배가 침몰한 사건
    유사도 0.556: 타이타닉은 1912년 4월 15일 빙산과 충돌해 침몰한 영국의 여객선이다.
    유사도 0.074: RAG는 검색과 생성을 결합한 자연어처리 기법이다.


In [9]:
# 7-2. documents 리스트에 문장 3개를 추가하고 이에 관한 질문으로 검색해보기
documents_1 = [
    "타이타닉은 1912년 4월 15일 빙산과 충돌해 침몰한 영국의 여객선이다.",
    "파이썬은 1991년 귀도 반 로섬이 개발한 프로그래밍 언어이다.",
    "BERT는 구글이 2018년에 발표한 자연어처리 모델이다.",
    "김치는 발효 채소를 이용한 한국의 전통 음식이다.",
    "RAG는 검색과 생성을 결합한 자연어처리 기법이다.",
    "딥러닝은 인공신경망을 여러 층으로 쌓아 학습하는 머신러닝의 한 분야이다.",

    "된장은 콩으로 만든 메주를 소금물에 발효시켜 만든다.",
    "컴퓨터는 사람이 명령을 내리면 인간보다 훨씬 빠르고 정확하게 계산을 대신해 주는 기계이다.",
    "핸드폰은 컴퓨터를 손바닥만 한 크기로 줄여서 들고 다닐 수 있게 만든 무선 통신 기기이다."
]

doc_embeddings_1 = model.encode(documents_1)

def search_1(query, top_k=2):
  query_vec = model.encode(query)
  scores = [cosine_sim(query_vec, doc_vec) for doc_vec in doc_embeddings_1]
  top_indices = np.argsort(scores)[::-1][:top_k]

  return [(documents_1[i], scores[i]) for i in top_indices]

query4 = "소프트웨어에는 시스템과 응용이 있다."
results4 = search_1(query4)
print(f'질문: {query4}')
for doc, score in results4:
  print(f'    유사도 {score:.3f}: {doc}')
print()

query5 = "전화, 문자 등을 바로 할 수 있다."
results5 = search_1(query5)
print(f'질문: {query5}')
for doc, score in results5:
  print(f'    유사도 {score:.3f}: {doc}')
print()

query6 = "쌈장은 쌈을 싸 먹을 때 쓰는 장을 말한다."
results6 = search_1(query6)
print(f'질문: {query6}')
for doc, score in results6:
  print(f'    유사도 {score:.3f}: {doc}')


질문: 소프트웨어에는 시스템과 응용이 있다.
    유사도 0.309: 컴퓨터는 사람이 명령을 내리면 인간보다 훨씬 빠르고 정확하게 계산을 대신해 주는 기계이다.
    유사도 0.268: BERT는 구글이 2018년에 발표한 자연어처리 모델이다.

질문: 전화, 문자 등을 바로 할 수 있다.
    유사도 0.652: 핸드폰은 컴퓨터를 손바닥만 한 크기로 줄여서 들고 다닐 수 있게 만든 무선 통신 기기이다.
    유사도 0.243: 컴퓨터는 사람이 명령을 내리면 인간보다 훨씬 빠르고 정확하게 계산을 대신해 주는 기계이다.

질문: 쌈장은 쌈을 싸 먹을 때 쓰는 장을 말한다.
    유사도 0.342: 된장은 콩으로 만든 메주를 소금물에 발효시켜 만든다.
    유사도 0.293: 김치는 발효 채소를 이용한 한국의 전통 음식이다.


In [10]:
# top_k를 1로 바꿔서 가장 관련 있는 문서가 1개만 나오는지 확인
def search_2(query, top_k=1):
  query_vec = model.encode(query)
  scores = [cosine_sim(query_vec, doc_vec) for doc_vec in doc_embeddings]
  top_indices = np.argsort(scores)[::-1][:top_k]

  return [(documents[i], scores[i]) for i in top_indices]

query1 = "자연어 처리 모델에는 뭐가 있어?"
results1 = search_2(query1)
print(f'질문: {query1}')
for doc, score in results1:
  print(f'    유사도 {score: .3f}: {doc}')

질문: 자연어 처리 모델에는 뭐가 있어?
    유사도  0.444: BERT는 구글이 2018년에 발표한 자연어처리 모델이다.
